In [1]:
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.ndimage import gaussian_filter1d

In [7]:
BASE = Path('/home/msp25gd/ResearchProjectMSc/ResolutionHandling')

# Prefer assessed-candidate export if present, otherwise fall back to the full dataset root.
SOURCE_CANDIDATES = [
    Path('/home/msp25gd/ResearchProjectMSc/HR/results/Assessed/Candidates'),
    Path('/home/msp25gd/Downloads/res/dataset'),
]

TARGET = BASE / 'processed_candidates'
LOG = BASE / 'logs'
TARGET.mkdir(parents=True, exist_ok=True)
LOG.mkdir(parents=True, exist_ok=True)

uniform_groups = np.load(BASE / 'uniform_groups.npy', allow_pickle=True).tolist()
mixed_groups = np.load(BASE / 'mixed_groups.npy', allow_pickle=True).tolist()
all_groups = sorted(uniform_groups + mixed_groups)

def _hit_count(base, groups, sample_size=200):
    sample = groups[:sample_size]
    return sum((base / str(g)).exists() for g in sample)

hit_map = {cand: _hit_count(cand, all_groups) for cand in SOURCE_CANDIDATES}
SOURCE = max(hit_map, key=hit_map.get)

if hit_map[SOURCE] == 0:
    raise FileNotFoundError(
        f'No candidate group folders found in any source base: {SOURCE_CANDIDATES}'
    )

print(f'Using SOURCE: {SOURCE} (sample hits: {hit_map[SOURCE]})')
print(f'Groups to process: {len(all_groups)}')
print(f'Uniform: {len(uniform_groups)} | Mixed: {len(mixed_groups)}')

Using SOURCE: /home/msp25gd/Downloads/res/dataset (sample hits: 200)
Groups to process: 497
Uniform: 333 | Mixed: 164


In [3]:
def sigma_pixels_for_resolution_degradation(R_current, R_target):
    # Kernel width in resolution units (dimensionless).
    if R_current <= R_target:
        return 0.0
    fwhm_current = 1.0 / float(R_current)
    fwhm_target = 1.0 / float(R_target)
    fwhm_kernel = (fwhm_target ** 2 - fwhm_current ** 2) ** 0.5
    sigma_resolution_units = fwhm_kernel / 2.354820045
    return max(0.0, sigma_resolution_units)

def degrade_one_spectrum(spec, R_current, R_target):
    # Convert from resolution units to pixel units using mean line width per pixel.
    sigma_r = sigma_pixels_for_resolution_degradation(R_current, R_target)
    if sigma_r <= 0:
        return spec
    sigma_pix = max(0.5, sigma_r * len(spec))
    return gaussian_filter1d(spec, sigma=sigma_pix, mode='nearest')

In [8]:
records = []

for group in all_groups:
    src = SOURCE / str(group)
    dst = TARGET / str(group)

    if not src.exists():
        records.append({'group': str(group), 'status': 'missing_source'})
        continue

    try:
        # Copy folder skeleton and everything first; then patch mixed groups.
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)

        if group in uniform_groups:
            records.append({'group': str(group), 'status': 'copied_uniform'})
            continue

        meta_path = dst / 'meta' / 'group_df.pkl'
        sk_path = dst / 'spec' / 'sK.npy'
        sh_path = dst / 'spec' / 'sH.npy'

        meta = pd.read_pickle(meta_path)
        if 'SPEC_RES' not in meta.columns:
            records.append({'group': str(group), 'status': 'missing_SPEC_RES'})
            continue

        rvals = pd.to_numeric(meta['SPEC_RES'], errors='coerce').to_numpy()
        if np.isnan(rvals).all():
            records.append({'group': str(group), 'status': 'all_SPEC_RES_nan'})
            continue

        r_target = float(np.nanmin(rvals))

        sK = np.load(sk_path)
        sH = np.load(sh_path)

        sK_new = np.array([degrade_one_spectrum(sK[i], rvals[i], r_target) for i in range(len(rvals))])
        sH_new = np.array([degrade_one_spectrum(sH[i], rvals[i], r_target) for i in range(len(rvals))])

        np.save(sk_path, sK_new)
        np.save(sh_path, sH_new)

        meta['SPEC_RES_original'] = meta['SPEC_RES']
        meta['SPEC_RES'] = r_target
        meta.to_pickle(meta_path)

        records.append({'group': str(group), 'status': 'degraded_mixed', 'target_resolution': r_target})

    except Exception as exc:
        records.append({'group': str(group), 'status': 'error', 'error': str(exc)})

log_df = pd.DataFrame(records)
log_df.to_csv(LOG / 'processed_groups_log.csv', index=False)
print(log_df['status'].value_counts(dropna=False).to_string())
print('\nSaved log:', LOG / 'processed_groups_log.csv')

status
copied_uniform    333
degraded_mixed    164

Saved log: /home/msp25gd/ResearchProjectMSc/ResolutionHandling/logs/processed_groups_log.csv
